# EPIC-KITCHENS 55 AVION Distractor Generation

This notebook uses the pre-trained AVION model to generate distractor options for Multiple Choice Questions (MCQs).
Instead of random options, we use the top-k predictions from the model to provide plausible distractors.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# AVION github repo
!git clone --recursive https://github.com/zhaoyue-zephyrus/AVION.git

Cloning into 'AVION'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 96 (delta 22), reused 18 (delta 18), pack-reused 52 (from 1)
Receiving objects: 100% (96/96), 1.38 MiB | 6.56 MiB/s, done.
Resolving deltas: 100% (31/31), done.
Submodule 'third_party/decord' (git@github.com:zhaoyue-zephyrus/decord-dev.git) registered for path 'third_party/decord'
Cloning into '/content/AVION/third_party/decord'...
Host key verification failed.
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.
fatal: clone of 'git@github.com:zhaoyue-zephyrus/decord-dev.git' into submodule path '/content/AVION/third_party/decord' failed
Failed to clone 'third_party/decord'. Retry scheduled
Cloning into '/content/AVION/third_party/decord'...
Host key verification failed.
fatal: Could not read from remote repository.

Please make sure you have t

In [ ]:
# Install Dependencies
!pip install einops kornia scikit-learn submitit timm transformers pandas
!pip install git+https://github.com/openai/CLIP.git
!pip install gdown
#!pip install "flash_attn==0.2.8"
!pip install flash_attn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.7 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-97ehjcr1
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-97ehjcr1
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=859541efa8ea2f29d3cf7248e8815a8547b62c9e009e94d8e8ef66d363775081
  Stored in directory: /tmp/pip-ephem-wheel-cache-7hhjyxf6/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 49.4 

In [ ]:
# Download checkpoint
import gdown
file_id = '1cVLsfjSHI0_7DeLKMjdHrSX-UKLmZKhE'
url = f'https://drive.google.com/uc?id={file_id}'

gdown.download(url, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1cVLsfjSHI0_7DeLKMjdHrSX-UKLmZKhE
From (redirected): https://drive.google.com/uc?id=1cVLsfjSHI0_7DeLKMjdHrSX-UKLmZKhE&confirm=t&uuid=3c0ef98f-dcdb-4d3d-a160-0a670ffa2586
To: /content/avion_finetune_mir_lavila_vitb_best.pt
100%|██████████| 1.79G/1.79G [00:23<00:00, 77.4MB/s]


'avion_finetune_mir_lavila_vitb_best.pt'

In [ ]:
import os
import sys
import json
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from collections import OrderedDict


AVION_ROOT = r"/content/AVION"
sys.path.insert(0, AVION_ROOT)

import avion.models.model_clip as model_clip
import clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
# Inputs
VIDEOS_TO_PROCESS = {
    "P01": [
        "P01_01", "P01_02", "P01_03", "P01_04", "P01_05",
        "P01_06", "P01_07", "P01_08", "P01_09", "P01_10",
        "P01_16", "P01_17", "P01_18", "P01_19"
    ],
    "P02": [
        "P02_01", "P02_02", "P02_03", "P02_04", "P02_05", "P02_06",
        "P02_07", "P02_08", "P02_09", "P02_10", "P02_11"
    ],
    "P03": [
        "P03_02", "P03_03", "P03_04", "P03_05", "P03_06", "P03_07",
        "P03_08", "P03_09", "P03_10", "P03_11", "P03_12", "P03_13",
        "P03_14", "P03_15", "P03_16", "P03_17", "P03_18", "P03_19", "P03_20",
        "P03_27", "P03_28"
    ],
    "P14": ["P14_01", "P14_02", "P14_03", "P14_04", "P14_05", "P14_07", "P14_09"]

}


CKPT_PATH = r"/content/avion_finetune_mir_lavila_vitb_best.pt"
FRAMES_ROOT = r"/content/EPIC-KITCHENS55"
ANNOTATIONS_PATH = r"/content/EPIC_train_action_labels.csv"

# Output
OUTPUT_JSONL = f"avion_distractors_{VIDEO_ID}.jsonl"

# Parameters
NUM_FRAMES = 16
TOP_K = 20

# Check paths
print(f"Checkpoint exists: {os.path.exists(CKPT_PATH)}")
print(f"Frames root exists: {os.path.exists(FRAMES_ROOT)}")
print(f"Annotations exist: {os.path.exists(ANNOTATIONS_PATH)}")

Checkpoint exists: True
Frames root exists: True
Annotations exist: True


In [ ]:
DRIVE_BASE = "/content/drive/MyDrive/EPIC-KITCHENS55"
for pid, videos in VIDEOS_TO_PROCESS.items():
    for vid in videos:

        # Define Paths
        src_tar = os.path.join(DRIVE_BASE, pid, f"{vid}.tar")
        dest_dir = os.path.join(FRAMES_ROOT, pid, "rgb", vid)

        if not os.path.exists(dest_dir):
            os.makedirs(dest_dir, exist_ok=True)

        # Copy & Extract
        if os.path.exists(src_tar):
            local_tar = os.path.join(dest_dir, f"{vid}.tar")
            os.system(f"cp {src_tar} {dest_dir}")
            ret = os.system(f"tar -xf {local_tar} -C {dest_dir}")
            else:
        else:
            print(f"Warning: Source tar not found at {src_tar}")


In [ ]:
# Load AVION Model
def load_avion_model(ckpt_path):

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    old_args = ckpt["args"]
    # Key cleaning: remove 'module.' prefix
    state_dict = OrderedDict((k.replace("module.", ""), v) for k, v in ckpt["state_dict"].items())

    model_name = getattr(old_args, 'model', 'Unknown')
    clip_length = getattr(old_args, 'clip_length', 16)
    use_fast_conv1 = getattr(old_args, 'use_fast_conv1', False)
    print(f"Model Config: {model_name}, Frames: {clip_length}, FastConv1: {use_fast_conv1}")

    model = model_clip.CLIP_VITB16(
        freeze_temperature=True,
        use_grad_checkpointing=False,
        use_bidirectional_lm=False,
        context_length=77,
        num_frames=clip_length,
        use_fast_conv1=use_fast_conv1,
        use_flash_attn=False,  # Cont use due to version mismatches
        project_embed_dim=256,
        pretrain_zoo="openai",
        pretrain_path=None,
    )

    # Remap keys to solve naming conflicts between different versions
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        new_k = k
        if "visual" in k:
            if "attn.Wqkv.weight" in k: new_k = k.replace("attn.Wqkv.weight", "attn.in_proj_weight")
            elif "attn.Wqkv.bias" in k: new_k = k.replace("attn.Wqkv.bias", "attn.in_proj_bias")
            elif "mlp.fc1.weight" in k: new_k = k.replace("mlp.fc1.weight", "mlp.c_fc.weight")
            elif "mlp.fc1.bias" in k: new_k = k.replace("mlp.fc1.bias", "mlp.c_fc.bias")
            elif "mlp.fc2.weight" in k: new_k = k.replace("mlp.fc2.weight", "mlp.c_proj.weight")
            elif "mlp.fc2.bias" in k: new_k = k.replace("mlp.fc2.bias", "mlp.c_proj.bias")

        new_state_dict[new_k] = v
    state_dict = new_state_dict

    # AVION might be trained on different number of input frames
    # They added a function in order to work with different number of frames
    from avion.models.utils import inflate_positional_embeds
    state_dict = inflate_positional_embeds(
        model.state_dict(),
        state_dict,
        num_frames=clip_length,
        load_temporal_fix='bilinear',
    )

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded state dict. Missing: {len(missing)}, Unexpected: {len(unexpected)}")

    model.to(DEVICE)
    model.eval()
    return model

model = load_avion_model(CKPT_PATH)

Model Config: CLIP_VITB16, Frames: 16, FastConv1: True
=> loading openai model
missing_keys:  ['logit_scale', 'visual.temporal_embedding', 'visual.image_projection', 'textual.text_projection']
unexpected_keys:  []
Remapping checkpoint keys to Standard Attention...
Inflating/checking positional embeddings...
Loaded state dict. Missing: 0, Unexpected: 0


In [ ]:
# Data Loading Helpers

def get_frame_path(root, participant_id, video_id, frame_idx):
    # Structure: root/P01/rgb/P01_01/frame_0000000001.jpg
    filename = f"frame_{frame_idx:010d}.jpg"
    return os.path.join(root, participant_id, "rgb", video_id, filename)

def sample_frame_indices(start_frame, stop_frame, n_frames=16):
    return np.linspace(start_frame, stop_frame, n_frames, dtype=int).tolist()

def load_and_preprocess_frames(frame_indices, root, participant_id, video_id):
    frames = []
    for idx in frame_indices:
        path = get_frame_path(root, participant_id, video_id, idx)
        if os.path.exists(path):
            frame = cv2.imread(path)
            if frame is not None:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)

    if len(frames) == 0:
        return None

    # Pad if missing frames
    while len(frames) < NUM_FRAMES:
        frames.append(frames[-1])

    frames_np = np.stack(frames)

    # Preprocess: [T, H, W, C] -> [1, C, T, H, W]
    size = 224
    x = torch.from_numpy(frames_np).float() / 255.0
    x = x.permute(0, 3, 1, 2) # [T, C, H, W]
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)

    # CLIP Normalization
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1).to(x.device)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1).to(x.device)
    x = (x.to(mean.device) - mean) / std

    x = x.permute(1, 0, 2, 3) # [C, T, H, W]
    x = x.unsqueeze(0)        # [1, C, T, H, W]
    return x

In [ ]:
# Compute Text Embeddings
full_df = pd.read_csv(ANNOTATIONS_PATH)
all_narrations = full_df['narration'].dropna().unique().tolist()
print(f"Found {len(all_narrations)} unique narrations in dataset.")

text_features_list = []
batch_size = 1000

model.eval()
with torch.no_grad():
    for i in tqdm(range(0, len(all_narrations), batch_size)):
        batch_texts = all_narrations[i : i + batch_size]
        text_input = clip.tokenize(batch_texts).to(DEVICE)
        text_embed = model.encode_text(text_input)
        text_embed = F.normalize(text_embed, dim=-1)
        text_features_list.append(text_embed.cpu())

all_text_features = torch.cat(text_features_list, dim=0).to(DEVICE)
print(f"\nEncoded text features shape: {all_text_features.shape}")

Loading dataset annotations...
Found 8793 unique narrations in dataset.
Encoding all narrations... (this may take a moment)


100%|██████████| 9/9 [00:16<00:00,  1.88s/it]

Encoded text features shape: torch.Size([8793, 256])


In [ ]:
for participant_id, video_list in VIDEOS_TO_PROCESS.items():
    for video_id in video_list:
        print(f"\nProcessing Video: {video_id} (Participant: {participant_id})")

        video_df = full_df[(full_df['participant_id'] == participant_id) & (full_df['video_id'] == video_id)]
        print(f"Found {len(video_df)} actions.")

        results_list = []
        for idx, row in tqdm(video_df.iterrows(), total=len(video_df), desc=f"Inferring {video_id}"):
            uid = int(row['uid'])
            start_frame = int(row['start_frame'])
            stop_frame = int(row['stop_frame'])
            narration = row['narration']

            # 1. Sample Frames
            frame_indices = sample_frame_indices(start_frame, stop_frame, NUM_FRAMES)

            # 2. Prepare Input
            video_input = load_and_preprocess_frames(frame_indices, FRAMES_ROOT, participant_id, video_id)

            if video_input is None:
                continue

            video_input = video_input.to(DEVICE)

            # 3. Model Inference
            with torch.no_grad():
                image_embed = model.encode_image(video_input)
                image_embed = F.normalize(image_embed, dim=-1)

                # Similarity
                logits = image_embed @ all_text_features.T
                logit_scale = model.logit_scale.exp()
                logits = logits * logit_scale
                probs = logits.softmax(dim=-1)

            # 4. Get Top-K
            top_probs, top_indices = probs[0].topk(TOP_K)

            distractors_with_conf = []
            for prob, idx in zip(top_probs, top_indices):
                idx = idx.item()
                text = all_narrations[idx]
                confidence = f"{prob.item():.4f}"
                distractors_with_conf.append({"answer": text, "confidence": confidence})

            # 5. Structure Output
            result_entry = {
                "uid": uid,
                "participant_id": participant_id,
                "video_id": video_id,
                "start_timestamp": row['start_timestamp'],
                "stop_timestamp": row['stop_timestamp'],
                "start_frame": start_frame,
                "stop_frame": stop_frame,
                "n_frames": NUM_FRAMES,
                "frame_indices": frame_indices,
                "ground_truth": {
                    "verb": int(row['verb_class']),
                    "verb_class": row['verb'],
                    "noun": int(row['noun_class']),
                    "noun_class": row['noun'],
                    "narration": narration
                },
                "distractors_with_confidence": distractors_with_conf
            }
            results_list.append(result_entry)

        # 6. Save per Video
        dest_dir = f"distactors/{participant_id}"
        if not os.path.exists(dest_dir):
            os.makedirs(dest_dir, exist_ok=True)
        output_filename = dest_dir + "/" + f"avion_distractors_{video_id}.jsonl"
        with open(output_filename, 'w') as f:
            for entry in results_list:
                f.write(json.dumps(entry) + '\n')
        print(f"Saved {len(results_list)} items to {output_filename}")


Processing Video: P01_01 (Participant: P01)
Found 326 actions.


Inferring P01_01: 100%|██████████| 326/326 [01:45<00:00,  3.10it/s]


Saved 326 items to distactors/P01/avion_distractors_P01_01.jsonl

Processing Video: P01_02 (Participant: P01)
Found 145 actions.


Inferring P01_02: 100%|██████████| 145/145 [00:48<00:00,  2.98it/s]


Saved 145 items to distactors/P01/avion_distractors_P01_02.jsonl

Processing Video: P01_03 (Participant: P01)
Found 42 actions.


...

Saved 57 items to distactors/P01/avion_distractors_P01_07.jsonl

Processing Video: P01_08 (Participant: P01)
Found 32 actions.


Inferring P01_08: 100%|██████████| 32/32 [00:10<00:00,  3.01it/s]


Saved 32 items to distactors/P01/avion_distractors_P01_08.jsonl

Processing Video: P01_09 (Participant: P01)
Found 879 actions.


Inferring P01_09:  70%|██████▉   | 612/879 [03:25<01:29,  2.98it/s]


KeyboardInterrupt: 